In [ ]:
!pip install -q transformers datasets kagglehub scikit-learn pynvml

In [ ]:
import os
import json
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_curve,
    auc
)

from transformers import (
    CLIPProcessor,
    CLIPModel
)

from torch.optim import AdamW


device = "cuda" if torch.cuda.is_available() else "cpu"

print("DEVICE:", device)

In [ ]:
import kagglehub


dataset_path = kagglehub.dataset_download(
    "parthplc/facebook-hateful-meme-dataset"
)

print(dataset_path)

In [ ]:
TRAIN_FILE = os.path.join(
    dataset_path,
    "data/train.jsonl"
)

DEV_FILE = os.path.join(
    dataset_path,
    "data/dev.jsonl"
)

TEST_FILE = os.path.join(
    dataset_path,
    "data/test.jsonl"
)

IMAGE_DIR = os.path.join(
    dataset_path,
    "data"
)

print(TRAIN_FILE)
print(DEV_FILE)
print(TEST_FILE)

In [ ]:
def load_jsonl(path):

    data = []

    with open(path, "r") as f:

        for line in f:

            data.append(
                json.loads(line)
            )

    return data


train_data = load_jsonl(TRAIN_FILE)

dev_data = load_jsonl(DEV_FILE)

test_data = load_jsonl(TEST_FILE)

print("TRAIN:", len(train_data))
print("DEV:", len(dev_data))
print("TEST:", len(test_data))

In [ ]:
train_data = train_data[:8500]

dev_data = dev_data[:500]

# test.jsonl has no labels
# use dev set for evaluation

print("TRAIN SAMPLES:", len(train_data))
print("DEV SAMPLES:", len(dev_data))

In [ ]:
sample = train_data[0]

image_path = os.path.join(
    IMAGE_DIR,
    sample["img"]
)

image = Image.open(image_path).convert("RGB")

plt.figure(figsize=(6,6))

plt.imshow(image)

plt.axis("off")

plt.title(f'LABEL: {sample["label"]}')

plt.show()

print(sample["text"])

In [ ]:
MODEL_ID = "openai/clip-vit-base-patch32"

processor = CLIPProcessor.from_pretrained(
    MODEL_ID
)

model = CLIPModel.from_pretrained(
    MODEL_ID
).to(device)

print("CLIP MODEL LOADED")

In [ ]:
class HatefulMemesDataset(Dataset):

    def __init__(
        self,
        data,
        processor,
        image_dir
    ):

        self.data = data

        self.processor = processor

        self.image_dir = image_dir


    def __len__(self):

        return len(self.data)


    def __getitem__(self, idx):

        sample = self.data[idx]


        image_path = os.path.join(
            self.image_dir,
            sample["img"]
        )


        image = Image.open(
            image_path
        ).convert("RGB")


        image = image.resize((224, 224))


        text = sample["text"]

        label = sample["label"]


        inputs = self.processor(

            text=[text],

            images=image,

            return_tensors="pt",

            padding="max_length",

            truncation=True,

            max_length=77
        )


        item = {

            "input_ids": inputs["input_ids"].squeeze(0),

            "attention_mask": inputs[
                "attention_mask"
            ].squeeze(0),

            "pixel_values": inputs[
                "pixel_values"
            ].squeeze(0),

            "label": torch.tensor(
                label,
                dtype=torch.long
            )
        }


        return item

In [ ]:
train_dataset = HatefulMemesDataset(

    train_data,

    processor,

    IMAGE_DIR
)


dev_dataset = HatefulMemesDataset(

    dev_data,

    processor,

    IMAGE_DIR
)

print("DATASETS READY")

In [ ]:
train_loader = DataLoader(

    train_dataset,

    batch_size=4,

    shuffle=True
)


dev_loader = DataLoader(

    dev_dataset,

    batch_size=4,

    shuffle=False
)

print("DATALOADERS READY")

In [ ]:
class CLIPClassifier(nn.Module):

    def __init__(self, clip_model):

        super().__init__()

        self.clip_model = clip_model


        self.classifier = nn.Sequential(

            nn.Linear(1024, 512),

            nn.ReLU(),

            nn.Dropout(0.2),

            nn.Linear(512, 2)
        )


    def forward(
        self,
        input_ids,
        attention_mask,
        pixel_values
    ):

        outputs = self.clip_model(

            input_ids=input_ids,

            attention_mask=attention_mask,

            pixel_values=pixel_values
        )


        image_features = outputs.image_embeds

        text_features = outputs.text_embeds


        combined = torch.cat(
            [image_features, text_features],
            dim=1
        )


        logits = self.classifier(combined)

        return logits

In [ ]:
classifier_model = CLIPClassifier(
    model
).to(device)

print("CLASSIFIER READY")

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = AdamW(

    classifier_model.parameters(),

    lr=2e-5
)

In [ ]:
GPU_AVAILABLE = torch.cuda.is_available()

if GPU_AVAILABLE:

    try:

        from pynvml import *

        nvmlInit()

        handle = nvmlDeviceGetHandleByIndex(0)

        ENERGY_MODE = "GPU"

    except:

        ENERGY_MODE = "CPU"

else:

    ENERGY_MODE = "CPU"



def get_power_usage():

    if ENERGY_MODE == "GPU":

        return (
            nvmlDeviceGetPowerUsage(handle)
            / 1000
        )

    else:

        return 65



def compute_energy_kwh(
    power_watts,
    duration_seconds
):

    return (
        power_watts *
        duration_seconds
    ) / (1000 * 3600)

In [ ]:
num_epochs = 10

classifier_model.train()

for epoch in range(num_epochs):

    running_loss = 0


    for batch in tqdm(train_loader):

        input_ids = batch[
            "input_ids"
        ].to(device)


        attention_mask = batch[
            "attention_mask"
        ].to(device)


        pixel_values = batch[
            "pixel_values"
        ].to(device)


        labels = batch[
            "label"
        ].to(device)


        optimizer.zero_grad()


        logits = classifier_model(

            input_ids,

            attention_mask,

            pixel_values
        )


        loss = criterion(
            logits,
            labels
        )


        loss.backward()

        optimizer.step()


        running_loss += loss.item()


    avg_loss = (
        running_loss /
        len(train_loader)
    )


    print(
        f"EPOCH {epoch+1} LOSS: {avg_loss:.4f}"
    )

In [ ]:
classifier_model.eval()


y_true = []

y_pred = []

y_scores = []

inference_times = []

energy_consumptions = []


with torch.no_grad():

    for batch in tqdm(dev_loader):

        input_ids = batch[
            "input_ids"
        ].to(device)


        attention_mask = batch[
            "attention_mask"
        ].to(device)


        pixel_values = batch[
            "pixel_values"
        ].to(device)


        labels = batch[
            "label"
        ].to(device)


        start_time = time.time()

        power = get_power_usage()


        logits = classifier_model(

            input_ids,

            attention_mask,

            pixel_values
        )


        end_time = time.time()


        duration = end_time - start_time


        energy = compute_energy_kwh(
            power,
            duration
        )


        probs = torch.softmax(
            logits,
            dim=1
        )


        predictions = torch.argmax(
            probs,
            dim=1
        )


        y_true.extend(
            labels.cpu().numpy()
        )


        y_pred.extend(
            predictions.cpu().numpy()
        )


        y_scores.extend(
            probs[:,1].cpu().numpy()
        )


        inference_times.append(duration)

        energy_consumptions.append(energy)

In [ ]:
accuracy = accuracy_score(
    y_true,
    y_pred
)


precision = precision_score(
    y_true,
    y_pred
)


recall = recall_score(
    y_true,
    y_pred
)


f1 = f1_score(
    y_true,
    y_pred
)


avg_time = np.mean(
    inference_times
)


avg_energy = np.mean(
    energy_consumptions
)


print("ACCURACY:", accuracy)

print("PRECISION:", precision)

print("RECALL:", recall)

print("F1-SCORE:", f1)

print("AVERAGE INFERENCE TIME (s):", avg_time)

print("AVERAGE ENERGY (kWh):", avg_energy)

In [ ]:
from sklearn.metrics import (

    accuracy_score,

    precision_score,

    recall_score,

    f1_score,

    confusion_matrix,

    classification_report
)


report = classification_report(

    y_true,

    y_pred,

    target_names=[

        "Non-Hateful",

        "Hateful"
    ],

    digits=4
)

print("\nCLASSIFICATION REPORT:\n")

print(report)

In [ ]:
cm = confusion_matrix(
    y_true,
    y_pred
)


plt.figure(figsize=(6,5))

plt.imshow(cm)

plt.colorbar()

plt.title("Confusion Matrix")

plt.xlabel("Predicted")

plt.ylabel("Actual")


for i in range(cm.shape[0]):

    for j in range(cm.shape[1]):

        plt.text(
            j,
            i,
            cm[i, j],
            ha="center",
            va="center"
        )


plt.show()

In [ ]:
fpr, tpr, thresholds = roc_curve(
    y_true,
    y_scores
)


roc_auc = auc(
    fpr,
    tpr
)


plt.figure(figsize=(6,5))

plt.plot(
    fpr,
    tpr,
    label=f"AUC = {roc_auc:.4f}"
)


plt.plot(
    [0,1],
    [0,1],
    linestyle="--"
)


plt.xlabel("False Positive Rate")

plt.ylabel("True Positive Rate")

plt.title("ROC Curve")

plt.legend()

plt.show()

In [ ]:
def perturb_text(text):

    text = text.lower()

    text = text.replace(
        "hate",
        "dislike"
    )

    text = text.replace(
        "kill",
        "harm"
    )

    return text


generalization_true = []

generalization_pred = []


classifier_model.eval()


with torch.no_grad():

    for sample in tqdm(dev_data[:20]):

        image_path = os.path.join(
            IMAGE_DIR,
            sample["img"]
        )


        image = Image.open(
            image_path
        ).convert("RGB")


        image = image.resize((224, 224))


        perturbed_text = perturb_text(
            sample["text"]
        )


        inputs = processor(

            text=[perturbed_text],

            images=image,

            return_tensors="pt",

            padding=True
        )


        input_ids = inputs[
            "input_ids"
        ].to(device)


        attention_mask = inputs[
            "attention_mask"
        ].to(device)


        pixel_values = inputs[
            "pixel_values"
        ].to(device)


        logits = classifier_model(

            input_ids,

            attention_mask,

            pixel_values
        )


        prediction = torch.argmax(
            logits,
            dim=1
        ).item()


        generalization_true.append(
            sample["label"]
        )


        generalization_pred.append(
            prediction
        )


generalization_accuracy = accuracy_score(
    generalization_true,
    generalization_pred
)


print(
    "GENERALIZATION ACCURACY:",
    generalization_accuracy
)

In [ ]:
results_df = pd.DataFrame({

    "Metric": [

        "Accuracy",

        "Precision",

        "Recall",

        "F1-score",

        "ROC-AUC",

        "Inference Time (s)",

        "Energy Consumption (kWh)",

        "Generalization Accuracy"
    ],


    "Value": [

        accuracy,

        precision,

        recall,

        f1,

        roc_auc,

        avg_time,

        avg_energy,

        generalization_accuracy
    ]
})


results_df

In [ ]:
results_df.to_csv(
    "clip_vit_b32_results.csv",
    index=False
)

print("RESULTS SAVED")

In [ ]:
torch.save(
    classifier_model.state_dict(),
    "clip_vit_b32_hateful_memes.pth"
)

print("MODEL SAVED")